# MPC / RMPC 통합 테스트 노트북 (8 케이스)

`MODE {MPC, RMPC}` × `sensor noise {off, on}` × `laser-power disturbance {off, on}` = **8 케이스**를 한 노트북에서 실행/비교/저장합니다.

| 케이스 | MODE | sensor noise | input disturbance |
|---|---|---|---|
| MPC_base / RMPC_base | MPC / RMPC | off | off |
| MPC_noise / RMPC_noise | MPC / RMPC | **on** | off |
| MPC_dist / RMPC_dist | MPC / RMPC | off | **on** |
| MPC_both / RMPC_both | MPC / RMPC | **on** | **on** |

전제
- **패치된 `GAMMA_MPC_temp_depth.py`** 를 이 노트북과 같은 폴더에 두세요 (입력 disturbance 인자 추가본).
- disturbance는 **컨트롤러 결정(레이저 명령값)에 더해져 plant에 들어가는 입력**을 교란합니다: `u_plant = u_cmd + d`.
- 기본은 **미측정 외란**(`disturbance_in_history=False`) — 컨트롤러는 d를 모름 → 진짜 외란 제거 성능 테스트.
- corner 임계값은 두 MODE 모두 **-0.95 통일**.


In [1]:
# ===== imports =====
import numpy as np
import cupy as cp
cp.cuda.Device(0).use()          # GAMMA가 쓰는 cupy 디바이스 (환경에 맞게 조정)
import pandas as pd
import sys, os, warnings, pickle, copy, csv

import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.nn import functional as F, ReLU

device = torch.device("cpu")     # 8케이스 공통 디바이스 (비교 공정성)
print("cuda available:", torch.cuda.is_available())

warnings.filterwarnings("ignore")
import logging; logging.disable(logging.CRITICAL)

sys.path.append('../1_model')

from TiDE import TideModule, quantile_loss, TiDE_forward
from RobustMPC_pytorch import RMPC
from nn_functions import surrogate
from torchmin import minimize as pytorch_minimize
from scipy.optimize import minimize, Bounds
from moving_average import moving_average_1d

from GAMMA_obj_temp_depth import GAMMA_obj
from obj_fun_depth import RMPC_obj_wo_constraint, RMPC_obj_wo_constraint_scipy
from GAMMA_MPC_temp_depth import GAMMA_MPC          # ← 패치본(입력 disturbance 지원)
from utils import sigmoid, LossParameters, GlobalState, softmax_max, softmax_min


cuda available: True


In [2]:
import torch
import pickle

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        return super().find_class(module, name)

import io

with open('TiDE_params_single_track_square_MV_temp_depth_less_cov_0915_w50_p50.pkl', 'rb') as f:
    nominal_params = CPU_Unpickler(f).load()

nominal_TiDE = nominal_params['model'].to("cpu")
P = 50
window = 50
TiDE = surrogate(nominal_params, nominal_TiDE)


# 레이저파워(원본 스케일) 범위 — disturbance 크기 잡을 때 참고
LP_MIN = float(TiDE.x_min[0][3]); LP_MAX = float(TiDE.x_max[0][3])
print(f"laser power range (W): {LP_MIN:.1f} ~ {LP_MAX:.1f}")


laser power range (W): 504.3 ~ 732.3


In [3]:
# ===== GAMMA config (인스턴스는 run_control 안에서 매번 새로 생성) =====
INPUT_DATA_DIR         = "data"
SIM_DIR_NAME           = "single_track_square"
BASE_LASER_FILE_DIR    = "laser_power_profiles/csv"
CLOUD_TARGET_BASE_PATH = "result"
solidus_temp = 1600
sim_interval = 5
init_runs    = 50

def build_fresh_gamma():
    """매 실행마다 깨끗한 GAMMA 시뮬레이터 + 초기 평균상태(init_avg) 반환."""
    g = GAMMA_obj(INPUT_DATA_DIR, SIM_DIR_NAME, BASE_LASER_FILE_DIR,
                  CLOUD_TARGET_BASE_PATH, solidus_temp, window, init_runs, sim_interval)
    init_avg = g.run_initial_steps()
    init_avg = torch.tensor(init_avg, dtype=torch.float32)[:, -window:]   # [2, 50]
    return g, init_avg


In [4]:
# ===== reference trajectory & fixed covariates =====
df_one_print = pd.read_csv('single_track_ref.csv')

loc_Z_list  = df_one_print["Z"].to_numpy().reshape(-1,1)
dist_X_list = df_one_print["Dist_to_nearest_X"].to_numpy().reshape(-1,1)
dist_Y_list = df_one_print["Dist_to_nearest_Y"].to_numpy().reshape(-1,1)

laser_power_ref  = torch.tensor(df_one_print["Laser_power"].to_numpy().reshape(-1,1), dtype=torch.float32)
laser_power_past = laser_power_ref[:window]

fix_covariates = torch.tensor(np.concatenate((loc_Z_list, dist_X_list, dist_Y_list), axis=1),
                              dtype=torch.float32)

mp_temp_raw = df_one_print["melt_pool_temperature"].to_numpy()
mp_temp_mv  = moving_average_1d(mp_temp_raw, 4)
mp_temp     = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp_ref = torch.tensor(mp_temp, dtype=torch.float32)


## 목적함수 팩토리 (MODE 토글, corner -0.95 통일)

In [5]:
# ===== objective factory =====
CORNER_TH = -0.95   # 두 모드 통일

def build_objective(MODE):
    if MODE == "MPC":
        lamda0, use_quantile = 0, False
    elif MODE == "RMPC":
        lamda0, use_quantile = 10, True
    else:
        raise ValueError(f"MODE must be 'MPC' or 'RMPC', got {MODE!r}")

    LossParam_local    = LossParameters(alpha0=1, delta_alpha=3, lamda0=lamda0)
    global_state_local = GlobalState()
    relu = ReLU()

    def obj(u_hat0, u_future_fix, u_past, x_past_fix, x_past, SP_hat, P, NN_Nominal):
        u_hat    = u_hat0.reshape(-1, 1)
        u_hat_in = u_hat0.unsqueeze(0)
        x_hat_all, x_hat_quantile = NN_Nominal.forward(u_hat, u_future_fix, u_past, x_past_fix, x_past)
        x_hat = x_hat_all[:, 0]

        u_hat_temp = u_hat_in[0, :, 0].reshape(-1, 1)
        u          = u_past[-1].reshape(-1, 1)
        u_hat1     = torch.concatenate((u, u_hat_temp))

        if x_past_fix[-1, 0] >= -0.2:
            with torch.no_grad():
                indices_1 = torch.where((u_future_fix[:, 1] <= CORNER_TH))[0]
                indices_2 = torch.where((u_future_fix[:, 2] <= CORNER_TH))[0]
                intersection      = torch.tensor(np.intersect1d(indices_1, indices_2))
                all_indices       = torch.arange(u_future_fix[:, 1].size(0))
                remaining_indices = all_indices[~torch.isin(all_indices, intersection)]

            if use_quantile:
                depth_upper = x_hat_quantile[:, 1, 2][remaining_indices].reshape(-1, 1)
                depth_lower = x_hat_quantile[:, 1, 0][remaining_indices].reshape(-1, 1)
            else:
                depth_med   = x_hat_all[:, 1][remaining_indices].reshape(-1, 1)
                depth_upper = depth_med
                depth_lower = depth_med

            g2 = (depth_upper - 0.412414) / 0.412414
            g3 = (0.1354 - depth_lower) / 0.1354

            Loss_g2 = (LossParam_local.lamda_1.item()*sum(relu(g2))
                       + 0.5*LossParam_local.alpha_1.item()*(sum(relu(g2))**2))
            LossParam_local.lamda_1 = LossParam_local.lamda_1 + LossParam_local.alpha_1*sum(relu(g2)).item()
            LossParam_local.alpha_1 = np.min((LossParam_local.alpha_1*LossParam_local.delta_alpha, 10000))

            Loss_g3 = (LossParam_local.lamda_2.item()*sum(relu(g3))
                       + 0.5*LossParam_local.alpha_2.item()*(sum(relu(g3))**2))
            LossParam_local.lamda_2 = LossParam_local.lamda_2 + LossParam_local.alpha_2*sum(relu(g3)).item()
            LossParam_local.alpha_2 = np.min((LossParam_local.alpha_2*LossParam_local.delta_alpha, 10000))

            if torch.any(u_future_fix[0,0]!=u_future_fix[:,0]) or torch.any(u_future_fix[0,0]!=x_past_fix[:,0]):
                Loss_g3 = 0

            Loss = Loss_g2/30 + Loss_g3/30
        else:
            Loss = 0

        Obj = (1*torch.sum((x_hat - torch.tensor(SP_hat.transpose(1,0), dtype=torch.float32))**2)
               + 10*torch.sum((u_hat1[:-1] - u_hat_temp)**2))

        if global_state_local.optim_iter_count == 0 or global_state_local.optim_iter_count == 300:
            global_state_local.update_f0(Obj.item())
        obj_s = Obj / global_state_local.get_f0()
        global_state_local.update_optim_iter_count()
        return obj_s + Loss / global_state_local.get_f0()

    return obj, LossParam_local, global_state_local


## 입력 disturbance 프리셋

`disturbance_fn(step, u_cmd, rng) -> d` (원본 스케일 [W]). 필요한 걸 골라 `DISTURBANCE`에 지정하세요.

In [6]:
# ===== disturbance presets =====
def dist_step(mag, t_on):
    """step >= t_on 부터 크기 mag[W]의 계단 외란."""
    return lambda step, u_cmd, rng: (mag if step >= t_on else 0.0)

def dist_pulse(mag, t_on, t_off):
    """[t_on, t_off) 구간에만 mag[W]."""
    return lambda step, u_cmd, rng: (mag if t_on <= step < t_off else 0.0)

def dist_bias(mag):
    """전 구간 상수 바이어스 mag[W]."""
    return lambda step, u_cmd, rng: mag

def dist_sine(amp, period, phase=0.0):
    """진폭 amp[W], 주기 period[step]의 정현파 외란."""
    return lambda step, u_cmd, rng: amp*np.sin(2*np.pi*step/period + phase)

def dist_gauss(std):
    """평균0, std[W] 가우시안 랜덤 외란 (재현성은 disturbance_seed)."""
    return lambda step, u_cmd, rng: float(rng.normal(0.0, std))

def dist_rel_bias(frac):
    """명령값 대비 비율 바이어스 (예: 0.10 = +10%)."""
    return lambda step, u_cmd, rng: frac*u_cmd


In [7]:
# ===== 공통 설정: 노이즈 / disturbance / seed =====
# --- sensor noise ---
NOISE_STD  = (20.0, 0.005)   # (temp[K], depth[mm]) 측정 노이즈 std
NOISE_SEED = 1000

# --- input(laser power) disturbance ---
# 크기는 위에서 출력된 LP 범위를 보고 조정하세요. (예: +50W 계단을 step 3000부터)
# DIST_MAG  = 50.0
# DIST_TON  = 3000
# DISTURBANCE = dist_step(DIST_MAG, DIST_TON)
DIST_STD    = 10.0                  # [W] 입력 외란 std (LP 범위 보고 조정)
DISTURBANCE = dist_gauss(DIST_STD)
DIST_SEED = 2000             # 랜덤 외란(dist_gauss 등) 쓸 때만 의미

N_STEP = 6196 - init_runs + 50


In [8]:
# ===== run helpers =====
def plot_fig(controller, N_step):
    plt.figure(figsize=[12,9])
    plt.subplot(3,1,1)
    plt.plot(controller.x_past_save[:N_step,0], label="GAMMA")
    plt.plot(controller.ref[:N_step], label="Reference")
    plt.ylabel("MP Temp (K)"); plt.legend()
    plt.subplot(3,1,2)
    plt.plot(controller.x_past_save[:N_step,1], label="GAMMA")
    plt.ylabel("MP Depth (mm)"); plt.legend()
    plt.subplot(3,1,3)
    plt.plot(controller.u_plant_save[:N_step], label="u_plant (실제 입력)")
    plt.plot(controller.u_cmd_save[:N_step], "--", label="u_cmd (명령)")
    plt.ylabel("Laser power (W)"); plt.xlabel("MPC step"); plt.legend()
    plt.tight_layout(); plt.show()


def run_control(MODE, *, add_meas_noise=False, meas_noise_std=(20.0,0.005), noise_seed=1000,
                add_input_disturbance=False, disturbance_fn=None, disturbance_seed=None,
                disturbance_in_history=False, clip_plant_input=False,
                N_step=None, plot_every=0, tag=None):
    GAMMA_class, init_avg = build_fresh_gamma()          # 매번 새 시뮬레이터
    if N_step is None:
        N_step = N_STEP
    obj, LossParam_local, global_state_local = build_objective(MODE)

    controller = GAMMA_MPC(GAMMA_class, TiDE, obj, mp_temp_ref,
                           window, P, fix_covariates, init_avg, laser_power_past,
                           LossParam_local, global_state_local,
                           add_meas_noise=add_meas_noise,
                           meas_noise_std=meas_noise_std, noise_seed=noise_seed,
                           add_input_disturbance=add_input_disturbance,
                           disturbance_fn=disturbance_fn, disturbance_seed=disturbance_seed,
                           disturbance_in_history=disturbance_in_history,
                           clip_plant_input=clip_plant_input)

    with open("iteration_log.csv", "w", newline="") as f:
        csv.writer(f).writerow(["timestep", "iterations", "type"])

    for i in tqdm(range(N_step), desc=tag or MODE):
        controller.MPC_run_one_step_pytorch()
        if plot_every and (i % plot_every == 0):
            plot_fig(controller, N_step)
    return controller


## 8 케이스 정의 & 실행

각 케이스를 개별 셀에서 돌릴 수 있고, 맨 아래 "전체 실행" 셀로 한 번에 돌릴 수도 있습니다.
결과는 `results[name]` 에 저장됩니다. (각 케이스는 full GAMMA 시뮬이라 시간이 걸립니다.)

In [9]:
# ===== 8 케이스 매트릭스 =====
CASES = {
    "MPC_base":   dict(MODE="MPC",  noise=False, dist=False),
    "RMPC_base":  dict(MODE="RMPC", noise=False, dist=False),
    "MPC_noise":  dict(MODE="MPC",  noise=True,  dist=False),
    "RMPC_noise": dict(MODE="RMPC", noise=True,  dist=False),
    "MPC_dist":   dict(MODE="MPC",  noise=False, dist=True),
    "RMPC_dist":  dict(MODE="RMPC", noise=False, dist=True),
    "MPC_both":   dict(MODE="MPC",  noise=True,  dist=True),
    "RMPC_both":  dict(MODE="RMPC", noise=True,  dist=True),
}

# ===== run_case (케이스 끝나면 즉시 CSV 저장) =====
results = {}

def save_result(controller, tag, n=6196):
    data = {
        'gamma_result_temp':  controller.x_past_save[:n,0].squeeze().numpy(),
        'gamma_result_depth': controller.x_past_save[:n,1].squeeze().numpy(),
        'ref':                np.asarray(controller.ref[:n]).squeeze(),
        'TiDE_pred_temp':     controller.NN_pred_save.detach().numpy()[:n,0],
        'TiDE_pred_depth':    controller.NN_pred_save.detach().numpy()[:n,1],
        'u_cmd':              controller.u_cmd_save[:n].squeeze().numpy(),
        'u_plant':            controller.u_plant_save[:n].squeeze().numpy(),
        'disturbance':        controller.disturbance_save[:n].squeeze().numpy(),
    }
    pd.DataFrame(data).to_csv(f'profile_{tag}.csv', index=False)
    print("saved", f'profile_{tag}.csv')

def run_case(name, save=True):
    c = CASES[name]
    ctrl = run_control(
        c["MODE"],
        add_meas_noise        = c["noise"],
        meas_noise_std        = NOISE_STD,
        noise_seed            = NOISE_SEED,
        add_input_disturbance = c["dist"],
        disturbance_fn        = DISTURBANCE if c["dist"] else None,
        disturbance_seed      = DIST_SEED,
        tag                   = name,
    )
    results[name] = ctrl
    if save:
        save_result(ctrl, name)          # ← 케이스 끝나자마자 저장
    print(f"[done] {name}")
    return ctrl

### ① disturbance 없음 (noise off)

In [10]:
# run_case("MPC_base")
# run_case("RMPC_base")

### ② sensor noise (disturbance off)

In [11]:
# run_case("MPC_noise")
# run_case("RMPC_noise")

### ③ laser-power disturbance (noise off)

In [12]:
# run_case("MPC_dist")
# run_case("RMPC_dist")
run_case("RMPC_both")

RMPC_both: 100%|██████████| 6196/6196 [36:03<00:00,  2.86it/s]  

saved profile_RMPC_both.csv
[done] RMPC_both


### ④ both (noise + disturbance)

In [13]:
# run_case("MPC_both")
run_case("RMPC_both")

RMPC_both:  17%|█▋        | 1061/6196 [05:11<25:08,  3.40it/s] 


KeyboardInterrupt: 

### ⟳ 전체 한 번에 실행 (선택)

In [ ]:
# 8케이스 전부 순차 실행. 시간이 오래 걸립니다.
for name in CASES:
    if name not in results:
        run_case(name)


## 📊 비교 & 요약

In [ ]:
# ===== 시나리오별 MPC vs RMPC 비교 (온도) =====
def temp(c, n=6196):  return c.x_past_save[:n,0].squeeze().numpy()
def depth(c, n=6196): return c.x_past_save[:n,1].squeeze().numpy()
def uplant(c, n=6196):return c.u_plant_save[:n].squeeze().numpy()

scenarios = [("base","disturbance 없음"), ("noise","sensor noise"),
             ("dist","input disturbance"), ("both","both")]

fig, axes = plt.subplots(2, 2, figsize=[15,9], sharex=True)
for ax,(key,title) in zip(axes.ravel(), scenarios):
    m, r = f"MPC_{key}", f"RMPC_{key}"
    if m in results:  ax.plot(temp(results[m]),  lw=0.8, label="MPC")
    if r in results:  ax.plot(temp(results[r]), lw=0.8, label="RMPC")
    if m in results:  ax.plot(results[m].ref[:6196], "k--", lw=0.7, label="ref")
    ax.set_title(title); ax.set_ylabel("MP Temp (K)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# ===== 요약 지표 테이블 =====
def rmse(a, b): return float(np.sqrt(np.nanmean((a-b)**2)))

rows = []
for name, c in results.items():
    n = 6196
    t  = temp(c, n); rf = np.asarray(c.ref[:n]).squeeze()
    dp = depth(c, n)
    up = uplant(c, n)
    rows.append({
        "case": name,
        "temp_RMSE_vs_ref": round(rmse(t, rf), 2),
        "temp_std":         round(float(np.nanstd(t)), 2),
        "depth_mean":       round(float(np.nanmean(dp)), 4),
        "depth_max":        round(float(np.nanmax(dp)), 4),
        "uplant_mean":      round(float(np.nanmean(up)), 1),
        "uplant_std":       round(float(np.nanstd(up)), 1),
    })
summary = pd.DataFrame(rows).set_index("case")
summary
